# Data inspection — least-cost path model

Run this notebook **before** running the model to verify that all required input files are present and correctly formatted.

Required files:
1. `data/output/cmr-mine-locations/all_mines_with_id.csv`
2. `data/output/cmr-construction-cost-friction-90m/cmr_friction_90m_clipped.tif`
3. `data/output/processed-protected-areas/cmr-protected-areas.gpkg`
4. `data/output/processed_roads_official/cleaned_merged_osm_heigit_liu-ver3-midterm.gpkg`
5. `data/output/processed-hansen-treecover/cameroon_treecover2024_30m.tif`
6. `data/input/damania-deforestation-lookup/damania_pct_cleared.csv`
7. `data/input/cmr_admin_boundaries_humdata/cmr_admin0_em.shp`
8. `data/input/cmr_admin_boundaries_humdata/cmr_admin2.shp`

If any files are missing, see the data preparation notebooks in `notebook/`.

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
import rasterio

## Required files checklist

In [2]:
REPO_ROOT = Path("..").resolve()

REQUIRED_FILES = {
    "Mine locations (CSV)":         REPO_ROOT / "data/output/cmr-mine-locations/all_mines_with_id.csv",
    "Construction cost friction":   REPO_ROOT / "data/output/cmr-construction-cost-friction-90m/cmr_friction_90m_clipped.tif",
    "Protected areas (GeoPackage)": REPO_ROOT / "data/output/processed-protected-areas/cmr-protected-areas.gpkg",
    "Road network (GeoPackage)":    REPO_ROOT / "data/output/processed_roads_official/cleaned_merged_osm_heigit_liu-ver3-midterm.gpkg",
    "Hansen treecover 2024":        REPO_ROOT / "data/output/processed-hansen-treecover/cameroon_treecover2024_30m.tif",
    "Damania deforestation lookup": REPO_ROOT / "data/input/damania-deforestation-lookup/damania_pct_cleared.csv",
    "Admin boundaries (level 0)":   REPO_ROOT / "data/input/cmr_admin_boundaries_humdata/cmr_admin0_em.shp",
    "Admin boundaries (level 2)":   REPO_ROOT / "data/input/cmr_admin_boundaries_humdata/cmr_admin2.shp",
}

all_ok = True
for label, path in REQUIRED_FILES.items():
    if path.exists():
        size_mb = path.stat().st_size / 1_000_000
        print(f"  ✓  {label:<40}  ({size_mb:6.1f} MB)  {path.name}")
    else:
        all_ok = False
        print(f"  ✗  {label:<40}  MISSING  {path}")

print()
n_found = sum(p.exists() for p in REQUIRED_FILES.values())
print(f"{n_found}/{len(REQUIRED_FILES)} files found")
if all_ok:
    print("All required files are present.")
else:
    print("Some files are missing. See data preparation notebooks in notebook/ to generate them.")

  ✓  Mine locations (CSV)                      (   0.0 MB)  all_mines_with_id.csv
  ✗  Construction cost friction                MISSING  D:\GitHub\_epa_thesis\strategic-road-planning\data\output\cmr-construction-cost-friction-90m\cmr_friction_90m_clipped.tif
  ✓  Protected areas (GeoPackage)              (   0.8 MB)  cmr-protected-areas.gpkg
  ✓  Road network (GeoPackage)                 ( 312.7 MB)  cleaned_merged_osm_heigit_liu-ver3-midterm.gpkg
  ✓  Hansen treecover 2024                     ( 277.6 MB)  cameroon_treecover2024_30m.tif
  ✓  Damania deforestation lookup              (   0.0 MB)  damania_pct_cleared.csv
  ✓  Admin boundaries (level 0)                (   0.3 MB)  cmr_admin0_em.shp
  ✓  Admin boundaries (level 2)                (   1.7 MB)  cmr_admin2.shp

7/8 files found
Some files are missing. See data preparation notebooks in notebook/ to generate them.


---
## 1) Mine locations

In [ ]:
mines_fp = REQUIRED_FILES["Mine locations (CSV)"]
mines_df = pd.read_csv(mines_fp)

print(f"Records: {len(mines_df)}")
print(f"\nMines by development stage:")
print(mines_df["DEV_STAGE_AGGREGATED_SNL"].value_counts())
mines_df[["ID", "PROP_NAME", "DEV_STAGE_AGGREGATED_SNL", "LATITUDE", "LONGITUDE"]].head(10)

In [ ]:
stage_colors = {"Early-stage": "#1f77b4", "Late-stage": "#ff7f0e", "Built": "#2ca02c"}

fig, ax = plt.subplots(figsize=(5, 6))
for stage, grp in mines_df.groupby("DEV_STAGE_AGGREGATED_SNL"):
    ax.scatter(grp["LONGITUDE"], grp["LATITUDE"],
               label=stage, color=stage_colors.get(stage, "gray"),
               s=50, edgecolors="white", linewidths=0.5)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Mine locations by development stage")
ax.legend(title="Stage", fontsize=8)
plt.tight_layout()
plt.show()

## 2) Road network

In [ ]:
roads_fp = REQUIRED_FILES["Road network (GeoPackage)"]
roads_gdf = gpd.read_file(roads_fp)

print(f"CRS: {roads_gdf.crs}")
print(f"Features: {len(roads_gdf)}")
print(f"\nRoad surface classification:")
print(roads_gdf["liu_surface"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
roads_gdf[roads_gdf["liu_surface"] == "unpaved"].plot(ax=ax, color="#aaaaaa", linewidth=0.2, label="Unpaved")
roads_gdf[roads_gdf["liu_surface"] == "paved"].plot(ax=ax, color="#cc3333", linewidth=0.5, label="Paved")
ax.set_title("Road network")
ax.legend(fontsize=8)
ax.axis("off")
plt.tight_layout()
plt.show()

## 3) Construction cost friction

In [ ]:
friction_fp = REQUIRED_FILES["Construction cost friction"]

with rasterio.open(friction_fp) as src:
    friction_arr = src.read(1)
    res_deg = src.res[0]
    friction_crs = src.crs

print(f"Shape: {friction_arr.shape}")
print(f"CRS: {friction_crs}")
print(f"Resolution: {res_deg:.6f}°  (~{res_deg * 111_000:.0f} m)")
finite = friction_arr[friction_arr < 1e8]
print(f"Cost range (passable pixels): [{finite.min():.0f}, {finite.max():.0f}] USD/pixel")

In [ ]:
step = 20
thumb = friction_arr[::step, ::step].astype(float)
thumb[thumb >= 1e8] = np.nan

fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(thumb, cmap="YlOrRd", aspect="auto")
plt.colorbar(im, ax=ax, label="USD/pixel", shrink=0.6)
ax.set_title(f"Construction cost friction (1:{step} downsample)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4) Protected areas

In [ ]:
pa_fp = REQUIRED_FILES["Protected areas (GeoPackage)"]
pa_gdf = gpd.read_file(pa_fp)

print(f"CRS: {pa_gdf.crs}")
print(f"Features: {len(pa_gdf)}")
print(f"Columns: {list(pa_gdf.columns)}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
pa_gdf.plot(ax=ax, color="#2d9e2d", alpha=0.5, edgecolor="none")
ax.set_title("Protected areas")
ax.axis("off")
plt.tight_layout()
plt.show()

## 5) Forest cover (Hansen treecover 2024)

In [ ]:
treecover_fp = REQUIRED_FILES["Hansen treecover 2024"]

with rasterio.open(treecover_fp) as src:
    tc_arr = src.read(1)
    tc_crs = src.crs

print(f"Shape: {tc_arr.shape}")
print(f"CRS: {tc_crs}")
print(f"Value range: [{tc_arr.min()}, {tc_arr.max()}]  (% tree cover, 0–100)")

In [ ]:
step = 10
fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(tc_arr[::step, ::step], cmap="Greens", aspect="auto", vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label="% tree cover", shrink=0.6)
ax.set_title(f"Hansen treecover 2024 (1:{step} downsample)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 6) Damania deforestation lookup

In [ ]:
damania_fp = REQUIRED_FILES["Damania deforestation lookup"]
damania_df = pd.read_csv(damania_fp)

print(f"Shape: {damania_df.shape}")
damania_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
quality_cols = [c for c in damania_df.columns if c != "distance_km"]
for col in quality_cols:
    label = col.replace("pct_cleared_", "").replace("_", " ").title()
    ax.plot(damania_df["distance_km"], damania_df[col], marker="o", markersize=4, label=label)
ax.set_xlabel("Distance from road (km)")
ax.set_ylabel("% forest cleared")
ax.set_title("Damania et al. deforestation distance-decay curve")
ax.legend(title="Governance quality", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7) Administrative boundaries

In [ ]:
admin0_fp = REQUIRED_FILES["Admin boundaries (level 0)"]
admin2_fp = REQUIRED_FILES["Admin boundaries (level 2)"]

admin0_gdf = gpd.read_file(admin0_fp)
admin2_gdf = gpd.read_file(admin2_fp)

print("National boundary (admin 0):")
print(f"  CRS: {admin0_gdf.crs}  |  Features: {len(admin0_gdf)}")
print("\nDepartments (admin 2):")
print(f"  CRS: {admin2_gdf.crs}  |  Features: {len(admin2_gdf)}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
admin2_gdf.plot(ax=ax, color="#f0f0f0", edgecolor="#aaaaaa", linewidth=0.5)
admin0_gdf.plot(ax=ax, color="none", edgecolor="#333333", linewidth=1.5)
ax.set_title("Cameroon administrative boundaries")
ax.axis("off")
plt.tight_layout()
plt.show()

## 8) Port locations

Port coordinates are hardcoded in the model (no input file required).

In [ ]:
port_locations = {
    "Kribi":  (2.9192, 9.8636),   # Deepwater port
    "Douala": (4.0500, 9.6972),   # Main port CMDLA
    "Limbe":  (4.0236, 9.2061),   # Minor port CMLIM
}

for name, (lat, lon) in port_locations.items():
    print(f"{name:8s}  lat={lat:.4f}  lon={lon:.4f}")